# Vivacity Physical Truth: free GPU compatibility probe

This notebook compares **three actual native MuJoCo and MJX-Warp steps** for each complete G1, Apollo and H1 embodiment. It does not train a policy or establish walking, carrying or cabinet-service success.

Use the free **T4 GPU / Standard RAM** runtime only. Do not buy compute units, upgrade a plan, mount Drive, start SSH or bypass a quota. If no free GPU is available, stop and keep the gate open. Download the reports before disconnecting. The complete cabinet task is still experimental.


In [ ]:
import os, sys, time, json, subprocess, pathlib, shutil, tarfile
pilot_started = time.monotonic()
pilot_deadline = pilot_started + 1800
print('Compatibility only; maximum 30 minutes. Python:', sys.version.split()[0])
assert sys.version_info >= (3, 11), 'Python 3.11+ is required'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], check=True, capture_output=True, text=True, timeout=20)
print(gpu.stdout)
os.environ.update(OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', JAX_PLATFORMS='cuda', XLA_PYTHON_CLIENT_PREALLOCATE='false')


In [ ]:
commit = 'fa47720a2437fed8687648e6f37f399d36af8987'
repo = pathlib.Path('/content/vivacity-physical-truth-pilot')
source = 'https://github.com/Tanish-Anand1/vivacity-investor-demo.git'
def command(args, *, cwd=None, budget=600):
    remaining = min(budget, int(pilot_deadline-time.monotonic()))
    if remaining <= 0: raise TimeoutError('The free compatibility pilot reached its 30-minute budget')
    return subprocess.run(args, cwd=cwd, check=True, timeout=remaining)
if not repo.exists():
    command(['git','clone','--filter=blob:none','--sparse','--no-checkout','--single-branch','--branch','codex/physical-truth-runtime',source,str(repo)])
    command(['git','sparse-checkout','set','vivacity','assets/menagerie','training'],cwd=repo)
    command(['git','checkout','--detach',commit],cwd=repo)
actual = subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
assert actual == commit, 'Existing pilot directory has a different source commit; do not overwrite it'
assert not subprocess.check_output(['git','diff','--name-only'],cwd=repo,text=True).strip(), 'Tracked source files changed'
output = repo/'data/free-gpu-pilot'
output.mkdir(parents=True,exist_ok=True)
(output/'commit.txt').write_text(actual+'\n')
(output/'gpu.txt').write_text(gpu.stdout)
env = pathlib.Path('/content/vivacity-physical-truth-venv')
if not env.exists(): command([sys.executable,'-m','venv',str(env)],budget=120)
python = str(env/'bin/python')
command([python,'-m','pip','install','--require-virtualenv','-r','training/requirements-gpu-lock.txt','--report',str(output/'resolver.json')],cwd=repo,budget=900)
command([python,'-m','pip','check'],cwd=repo,budget=30)
(output/'installed.txt').write_bytes(subprocess.check_output([python,'-m','pip','freeze'],timeout=30))


In [ ]:
outcomes=[]
output=repo/'data/free-gpu-pilot'/('attempt-'+str(time.time_ns()))
output.mkdir(parents=True,exist_ok=False)
(output/'commit.txt').write_text(commit+'\n')
(output/'gpu.txt').write_text(gpu.stdout)
for robot in ('h1','g1','apollo'):
    print('\nRunning native/MJX compatibility:',robot,flush=True)
    path=output/(robot+'.json')
    try:
        command([python,'-u','-m','training.preflight','--robot',robot,'--steps','3','--implementation','warp','--output',str(path)],cwd=repo,budget=600)
    except (subprocess.CalledProcessError,subprocess.TimeoutExpired,TimeoutError) as error:
        if not path.exists(): path.write_text(json.dumps({'robot':robot,'passed':False,'failure_type':type(error).__name__,'failure':str(error),'trained_policy':None},indent=2))
    report=json.loads(path.read_text())
    outcomes.append({key:report.get(key) for key in ('robot','passed','failure_type','failure','native_vs_mjx','compiled_model_sha256','trained_policy')})
summary={'scope':'GPU compatibility only; not task acceptance or training','source_commit':commit,'elapsed_s':time.monotonic()-pilot_started,'outcomes':outcomes,'all_passed':all(item['passed'] for item in outcomes)}
(output/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps(summary,indent=2))
archive=pathlib.Path('/content/vivacity-free-gpu-evidence.tar.gz')
with tarfile.open(archive,'w:gz') as tar: tar.add(output,arcname='free-gpu-pilot')
print('Evidence archive:',archive)


In [ ]:
# Run after reviewing the measured report. Then use Runtime > Disconnect and delete runtime.
from google.colab import files
files.download(str(archive))
